# JacobianFactor

A `JacobianFactor` in GTSAM represents a linear factor in terms of a Jacobian matrix $A$ (or matrices $A_j$) and a right-hand side vector $b$. It is commonly produced by linearizing a `NonlinearFactor` at a given linearization point. `JacobianFactor` is a key building block for constructing Gaussian (linear) factor graphs, which are solved in each iteration of nonlinear optimization.

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab  # type: ignore
    %pip install --quiet gtsam-develop
except Exception:
    pass  # Not in Colab

## Mathematical Form

A Jacobian factor encodes a linear relationship of the form:

$$
A x = b + \text{noise}
$$

where $A$ is the Jacobian matrix, $x$ is the stacked vector of variables, and $b$ is the measurement vector. The error is typically:

$$
\text{error}(x) = \frac{1}{2} \|A x - b\|^2
$$

If there are multiple variables $x_j$, the Jacobian factor can be written as:

$$
A_1 x_1 + A_2 x_2 + \cdots + A_n x_n = b + \text{noise}
$$

where each $A_j$ is the Jacobian block corresponding to variable $x_j$. The error becomes:

$$
\text{error}(x_1, \ldots, x_n) = \frac{1}{2} \left\| \sum_{j=1}^n A_j x_j - b \right\|^2
$$

This block structure allows the factor to represent relationships between multiple variables in the factor graph.

In [ ]:
# Import required libraries
import gtsam
import numpy as np

## Creating a JacobianFactor

We create a simple `JacobianFactor` for two variables. The Jacobian matrix $A$ and measurement vector $b$ are specified as numpy arrays.

In [ ]:
# Define variable keys
key1 = 0
key2 = 1

# Define Jacobian blocks and measurement
A1 = np.array([[1.0, 0.0]])  # 1x2 block for key1
A2 = np.array([[0.0, 1.0]])  # 1x2 block for key2
b = np.array([2.0])          # 1D measurement
noise = gtsam.noiseModel.Unit.Create(1)

# Create the JacobianFactor
factor = gtsam.JacobianFactor(key1, A1, key2, A2, b, noise)

# Add to a GaussianFactorGraph
graph = gtsam.GaussianFactorGraph()
graph.add(factor)
print(f"Graph size after adding factor: {graph.size()}")

## Accessing Factor Properties

We can inspect the keys, print the factor, and examine its Jacobian blocks.

In [ ]:
factor.getA()

In [ ]:
# Print the factor
graph.at(0).print()

# Get keys
print(f"Keys: {list(factor.keys())}")

# Get Jacobian blocks
for i, key in enumerate(factor.keys()):
    block = factor.getA()
    print(f"Jacobian block for key {key}:\n{block}")

## Evaluating Error

We can compute the error for a given assignment of variable values using a `gtsam.Values` object.

In [ ]:
# Create Values and insert assignments
values = gtsam.VectorValues()
values.insert(key1, np.array([2.0, 0.0])) # type: ignore
values.insert(key2, np.array([0.0, 2.0])) # type: ignore

# Compute error
err = factor.error(values)
print(f"Error at given values: {err}")

## Visualization

If Graphviz is available, we can visualize the factor graph.

In [ ]:
import graphviz

dot_str = graph.dot()
graphviz.Source(dot_str)